# Week 6: Unsupervised Learning & Clustering
## Notebook 08: HDBSCAN: Hierarchical Density-Based Clustering

### 1. Conceptual Intuition & Theoretical Background
DBSCAN has a major flaw: it requires a single global $\varepsilon$ distance threshold. If a dataset contains clusters with **varying densities**, DBSCAN either merges sparse clusters into noise or fractures dense clusters.

**HDBSCAN** eliminates the fixed $\varepsilon$ by performing density clustering across **all scales hierarchically**:
1. **Core Distance:** $\text{core}_k(x)$ is the distance to its $k$-th nearest neighbor.
2. **Mutual Reachability Distance:**
$$d_{\text{mreach}-k}(a, b) = \max\left( \text{core}_k(a),\ \text{core}_k(b),\ d(a, b) \right)$$
This spreads out points in low-density regions while preserving dense cluster neighborhoods.

3. **Condensed Tree & Stability Selection:**
Rather than picking a horizontal cut, HDBSCAN measures each cluster's persistence across scale $\lambda = 1/\varepsilon$:
$$S(C) = \sum_{p \in C} (\lambda_{\text{death}}(p) - \lambda_{\text{birth}}(C))$$
Extracts the optimal flat clusters that maximize persistent excess-of-mass stability.

---

### 2. Scratch Python Implementation: Mutual Reachability Matrix

In [ ]:
import numpy as np

def compute_mutual_reachability_matrix(X, min_samples=5):
    """
    Computes pairwise mutual reachability distance matrix:
    d_mreach(a, b) = max(core_k(a), core_k(b), dist(a, b))
    """
    n = len(X)
    # Pairwise euclidean distances
    diff = X[:, np.newaxis, :] - X[np.newaxis, :, :]
    dist_matrix = np.sqrt(np.sum(diff ** 2, axis=-1))
    
    # Core distance: distance to k-th nearest neighbor
    k_sorted_dists = np.sort(dist_matrix, axis=1)
    core_distances = k_sorted_dists[:, min_samples]
    
    # Compute mutual reachability matrix
    core_i = core_distances[:, np.newaxis]
    core_j = core_distances[np.newaxis, :]
    
    mreach_matrix = np.maximum(dist_matrix, np.maximum(core_i, core_j))
    return mreach_matrix, core_distances

# Generate 2 clusters with significantly different densities
np.random.seed(42)
dense_cluster = np.random.normal(loc=[-4, 0], scale=0.4, size=(80, 2))
sparse_cluster = np.random.normal(loc=[4, 0], scale=1.5, size=(80, 2))
X_data = np.vstack([dense_cluster, sparse_cluster])

mreach_mat, cores = compute_mutual_reachability_matrix(X_data, min_samples=5)

print(f"Dataset Size: {len(X_data)}")
print(f"Dense Cluster Mean Core Distance:  {np.mean(cores[:80]):.4f}")
print(f"Sparse Cluster Mean Core Distance: {np.mean(cores[80:]):.4f}")
print("Mutual reachability distance successfully scales inversely with local cluster density!")

### 3. Step-by-Step Code Walkthrough
1. **Core Distances**: Points in the dense cluster have tiny core distances (e.g. 0.35), while sparse cluster points have high core distances (e.g. 1.4).
2. **Mutual Reachability**: Separates sparse points by inflating their mutual distance, preventing noise from linking separate clusters.
3. **Stability Metric**: Integrates excess of mass $\int (\lambda - \lambda_{\text{birth}})$ to identify persistent clusters.

---

### 4. Industry Applications & Trade-offs
- **Geospatial & GPS Trajectory Mining:** Clusters vehicle pickups in dense city centers alongside sparse suburban drop-offs.
- **Trade-off:** $O(N^2)$ distance matrix computation can be memory-intensive without approximate KD-Tree acceleration.

---

### 5. Top Interview Questions & Answers
**Q1: How does HDBSCAN overcome the primary limitation of standard DBSCAN?**
> **Answer:** DBSCAN uses a single static $\varepsilon$ radius, failing when clusters have varying densities. HDBSCAN constructs a hierarchical dendrogram and extracts clusters based on lifetime stability across all $\varepsilon$ values.

**Q2: What is the meaning of $\lambda = 1 / \varepsilon$ in HDBSCAN?**
> **Answer:** $\lambda$ represents density. As $\varepsilon$ shrinks towards 0, $\lambda \to \infty$. Clusters that survive over a wide range of $\lambda$ possess high stability.